**Component 2 - Implementation**

This is the route-finding system (Uniform Cost Search), designed in Component 1, implemented in Python. The program searches the campus graph and calculates the lowest-cost route between two campus locations given as arguments. This follows the pseudocode detailed in Component 1, using a priority queue, explored set, best-cost dictionary and expanded list.

In [ ]:
import heapq

#Below is the implementation of the campus graph, using a dictionary as an edge table.
#Each key is a node, containing an inner dictionary which maps the node's neighbours
#to its walking costs.

campusGraph = {
    "WPT": {"SL": 3},
    "SL":  {"WPT": 3, "WLF": 4, "SDD": 2},
    "WLF": {"SL": 4, "SDD": 4},
    "SDD": {"WLF": 4, "PL": 5},
    "PL":  {"SDD": 5, "H": 5, "MC": 6, "SCH": 4},
    "H":   {"PL": 5, "SCH": 2, "SRM": 3, "SU": 4},
    "SRM": {"H": 3, "SCH": 2, "CW": 5},
    "CW":  {"SRM": 5, "SU": 6, "SDW": 2},
    "SDW": {"CW": 2, "MC": 5},
    "MC":  {"SDW": 5, "SU": 5, "PL": 6},
    "SU":  {"MC": 5, "CW": 6, "H": 4},
    "SCH": {"PL": 4, "H": 2, "SRM": 2},
}

locationNames = {
    "WPT": "West Park Teaching Hub",
    "SL": "STEMLab",
    "WLF": "Wolfson",
    "SDD": "Sir David Davies",
    "PL": "Pilkington Library",
    "H": "Haslegrave",
    "SCH": "Schofield",
    "SRM": "Sir Richard Morris",
    "SU": "Student Union",
    "MC": "Medical Centre",
    "CW": "Clyde Williams",
    "SDW": "Sir David Wallace Sports Hall"
}


In [ ]:
def uniform_cost_search(graph, start, goal):
  frontier = [(0, start, [start])] #this holds tuple (cost, node, path) which is ordered by cost
  best_cost = {start: 0} #tracks cheapest known cost
  expanded = [] #this tracks and orders the popping of nodes
  explored = set() #this tracks the nodes which have been fully explored

  while frontier:
    current_cost, current_node, path = heapq.heappop(frontier)


    if current_node in explored:  #skip if already expanded with a cheaper path

      continue

    expanded.append(current_node)

    if current_node == goal: #check goal when popping, not when pushing, so the cheapest path is returned

      return path, current_cost, expanded

    explored.add(current_node)

    for neighbour, edge_cost in graph[current_node].items():
      if neighbour in explored:
        continue

      updated_cost = current_cost + edge_cost

      #if cheapest route for neighbour, then push

      if neighbour not in best_cost or updated_cost < best_cost[neighbour]:
        best_cost[neighbour] = updated_cost
        updated_path = path + [neighbour]
        heapq.heappush(frontier, (updated_cost, neighbour, updated_path))

  return None, None, expanded


def display_route(start, goal):

  path, cost, expanded = uniform_cost_search(campusGraph, start, goal)

  startLocationName = locationNames.get(start, start)
  goalLocationName = locationNames.get(goal, goal)

  print("Start location: "+startLocationName)
  print("Goal location: "+goalLocationName)
  print()

  if path == None:
    print("No route exists between chosen start and goal.")
  else:
   print(f"Route: {' -> '.join(path)}\nCost: {cost} minutes")
   print(f"Expanded nodes in order: {', '.join(expanded)}")

  print("\n")


In [ ]:
#Test cases

#Standard test cases

#Long route (spans most of campus)
#Expected Route: WPT -> SL -> SDD -> PL -> MC -> SDW, Expected Cost: 21 minutes
display_route("WPT", "SDW")

#Mid-length
#Expected Route: SRM -> SCH -> PL -> SDD, Expected Cost: 11 minutes
display_route("SRM", "SDD")

#Short route (between neighbours)
#Expected Route: H -> SU, Expected Cost: 4 minutes
display_route("H", "SU")

#Edge cases

#One way edge (in direction of the edge)
#Expected Route: SL -> SDD, Expected Cost: 2 minutes
display_route("SL", "SDD")

#One way edge (other direction, forced detour due to edge)
#Expected Route: SDD -> WLF -> SL, Expected Cost: 8 minutes
display_route("SDD", "SL")

#Tied-cost path
#SRM -> MC has three routes, each costing 12 minutes
#The three distinct routes are (SRM -> CW -> SDW -> MC), (SRM -> H -> SU -> MC), (SRM -> SCH -> PL -> MC).
#UCS is determinstic therefore the same route should be consistently chosen.
#Expected Route: SRM -> SCH -> PL -> MC, Expected Cost: 12 minutes
display_route("SRM", "MC")

#Start location equals goal location
#Expected Route: SL, Expected Cost: 0 minutes
display_route("SL", "SL")




Start location: West Park Teaching Hub
Goal location: Sir David Wallace Sports Hall

Route: WPT -> SL -> SDD -> PL -> MC -> SDW
Cost: 21 minutes
Expanded nodes in order: WPT, SL, SDD, WLF, PL, SCH, H, MC, SRM, SU, CW, SDW


Start location: Sir Richard Morris
Goal location: Sir David Davies

Route: SRM -> SCH -> PL -> SDD
Cost: 11 minutes
Expanded nodes in order: SRM, SCH, H, CW, PL, SDW, SU, SDD


Start location: Haslegrave
Goal location: Students’ Union

Route: H -> SU
Cost: 4 minutes
Expanded nodes in order: H, SCH, SRM, SU


Start location: STEMLab
Goal location: Sir David Davies

Route: SL -> SDD
Cost: 2 minutes
Expanded nodes in order: SL, SDD


Start location: Sir David Davies
Goal location: STEMLab

Route: SDD -> WLF -> SL
Cost: 8 minutes
Expanded nodes in order: SDD, WLF, PL, SL


Start location: Sir Richard Morris
Goal location: Medical Centre

Route: SRM -> SCH -> PL -> MC
Cost: 12 minutes
Expanded nodes in order: SRM, SCH, H, CW, PL, SDW, SU, SDD, MC


Start location: STEMLa

**Observations**

The results of each of the seven tests match their expected ouput, confirming in each case the lowest-cost rote is found. The expanded list helps us analyse the behaviour and decisions of the UCS algorithm.

Notes on behaviour:

Short routes still expand multiple nodes, like H -> SU, which expands four nodes to explore cheaper-cost neighbour before goal-checking.

The one-way tests show the algorithm deals with asymmetry, as they give different routes and costs for the same pair of nodes.

Tied-cost shows the algorithm's deterministic behaviour as it will always return the same route for the same input of nodes.